In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Create widgets for landing_timestamp and table_name
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

# Get widget values into variables
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print(f"landing_timestamp: {landing_timestamp}")
print(f"table_name: {table_name}")
print(f"merge_keys: {merge_keys}")
print(f"load_type: {load_type}")

In [0]:
bronze_df = spark.read.table(f"adb_caresync.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

In [0]:
updated_df = bronze_df.withColumn("insert_timestamp",current_timestamp() ).withColumn( "update_timestamp" , current_timestamp())

In [0]:
target_table = f"adb_caresync.silver.{table_name}"

print(f"=" * 80)
print(f"LOAD CONFIGURATION")
print(f"=" * 80)
print(f"Target Table  : {target_table}")
print(f"Load Type     : {load_type}")
print(f"Source Records: {updated_df.count()}")
print(f"Source Columns: {updated_df.columns}")
print(f"=" * 80)

if load_type == "FULL":
    print(f"\n[FULL LOAD] Overwriting target table: {target_table}")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print(f"[FULL LOAD] Successfully overwrote {target_table}")

elif load_type == "APPEND":
    print(f"\n[APPEND] Appending data to target table: {target_table}")
    updated_df.write.mode("append").saveAsTable(target_table)
    print(f"[APPEND] Successfully appended data to {target_table}")

elif load_type == "MERGE":
    merge_key_list = [k.strip() for k in merge_keys.split(",")]
    merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in merge_key_list])

    print(f"\n[MERGE] Merge Keys     : {merge_key_list}")
    print(f"[MERGE] Merge Condition: {merge_condition}")

    table_exists = spark.catalog.tableExists(target_table)
    print(f"[MERGE] Target table exists: {table_exists}")

    if table_exists:
        updated_df.createOrReplaceTempView("source_view")
        print(f"[MERGE] Created temp view: source_view")

        # Preserve original insert_timestamp for existing records; update everything else
        update_cols = [c for c in updated_df.columns if c != "insert_timestamp"]
        update_set = ", ".join([f"target.{c} = source.{c}" for c in update_cols])

        insert_cols = ", ".join(updated_df.columns)
        insert_vals = ", ".join([f"source.{c}" for c in updated_df.columns])

        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING source_view AS source
            ON {merge_condition}
            WHEN MATCHED THEN UPDATE SET {update_set}
            WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
        """

        print(f"\n[MERGE] Generated SQL:")
        print(f"-" * 80)
        print(merge_sql)
        print(f"-" * 80)

        spark.sql(merge_sql)
        print(f"\n[MERGE] Successfully merged data into {target_table}")
    else:
        print(f"[MERGE] Target table does not exist — performing initial full load")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"[MERGE] Successfully created {target_table} with initial data")

else:
    raise ValueError(f"Unsupported load_type: '{load_type}'. Expected 'FULL', 'APPEND', or 'MERGE'.")

print(f"\n{'=' * 80}")
print(f"LOAD COMPLETE")
print(f"{'=' * 80}")

In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))